In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

# load dataset
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

print("dataset shape:", df.shape)
print(df.head())
print("\nmissing values:\n", df.isnull().sum())

dataset shape: (177990, 14)
         date  year  month  day         region    country category  \
0  2025-10-12  2025     10   12    Middle East        UAE  Chronic   
1  2020-09-15  2020      9   15     South Asia      India  Chronic   
2  2020-02-26  2020      2   26    Middle East        UAE  Vitamin   
3  2025-11-09  2025     11    9     South Asia  Sri Lanka  Chronic   
4  2022-04-04  2022      4    4  South America  Argentina  Chronic   

     medicine age_group  units_sold  unit_price  stock_level  \
0  Amlodipine      0-12         328       45.38         4774   
1  Amlodipine     26-45         371       75.49         4584   
2   Vitamin C      0-12         948       22.51         3934   
3  Amlodipine      0-12         275       63.29         4544   
4  Amlodipine     26-45         563       44.28         4284   

   expiry_days_remaining  covid_flag  
0                    466           0  
1                    181           1  
2                    556           1  
3         

In [2]:
# convert date into useful calendar features
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["day_of_week"] = df["date"].dt.dayofweek
df["quarter"] = df["date"].dt.quarter
df = df.drop(columns=["date"])

# classify units sold as high or low using the median
median_sales = df["units_sold"].median()

df["sales_category"] = np.where(
    df["units_sold"] >= median_sales, "High", "Low"
)

# remove the original target and prepare features
X = df.drop(columns=["units_sold", "sales_category"])
y = df["sales_category"]

print("median units sold:", median_sales)
print("\ntarget distribution:\n", y.value_counts())

median units sold: 387.0

target distribution:
 sales_category
High    89217
Low     88773
Name: count, dtype: int64


In [3]:
# split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# identify numerical and categorical columns
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# handle missing values and encode categorical features
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("training samples:", X_train_processed.shape[0])
print("testing samples:", X_test_processed.shape[0])
print("encoded features:", len(feature_names))

training samples: 142392
testing samples: 35598
encoded features: 56


In [4]:
# use Extra Trees to rank feature importance
selector_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

selector_model.fit(X_train_processed, y_train)

# select the 20 most important features
importance = selector_model.feature_importances_
top_indices = np.argsort(importance)[-20:]

X_train_selected = X_train_processed[:, top_indices]
X_test_selected = X_test_processed[:, top_indices]
selected_features = feature_names[top_indices]

print("selected features:")
print(selected_features)

selected features:
['cat__medicine_Vitamin C' 'cat__medicine_Amoxicillin'
 'cat__category_Chronic' 'cat__medicine_Metformin'
 'cat__category_Antibiotic' 'cat__category_Vitamin' 'cat__age_group_65+'
 'cat__age_group_13-25' 'cat__age_group_46-65' 'cat__age_group_0-12'
 'cat__age_group_26-45' 'num__day_of_week' 'num__day'
 'num__expiry_days_remaining' 'num__unit_price' 'num__year'
 'num__covid_flag' 'num__month' 'num__quarter' 'num__stock_level']


In [ ]:
# initialize Random Forest with OOB estimation
rf = RandomForestClassifier(
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

# define hyperparameter search space
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

# search for the best combination
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=12,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

random_search.fit(X_train_selected, y_train)

# retrieve optimized model
best_rf = random_search.best_estimator_

print("best parameters:")
print(random_search.best_params_)

print("\nbest cross-validation F1-score:")
print(round(random_search.best_score_, 4))

print("\nout-of-bag accuracy estimate:")
print(round(best_rf.oob_score_, 4))

print(round(1 - best_rf.oob_score_, 4))

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [ ]:
# predict on unseen test data
y_pred = best_rf.predict(X_test_selected)

# calculate performance metrics
print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("precision:", round(precision_score(y_test, y_pred, pos_label="High"), 4))
print("recall:", round(recall_score(y_test, y_pred, pos_label="High"), 4))
print("f1-score:", round(f1_score(y_test, y_pred, pos_label="High"), 4))

print("\nclassification report:\n")
print(classification_report(y_test, y_pred))

# display confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    cmap="Blues",
    values_format="d"
)

plt.title("random forest confusion matrix")
plt.show()